# Analysis — appearance vs. geometry

Reads `results/results.csv` (all 257 runs, written by `scripts/evaluate.py`) and rebuilds
the analysis tables and figures, without a GPU. Set the `RESULTS_CSV` environment variable
to read a different results file.

The primary geometry number is **unaligned** depth MAE on NeRF-Synthetic (alignment would
erase the calibration errors the corruption stages inject) and the official Chamfer distance
on DTU. Deltas are against vanilla 3DGS — random initialization, no depth — at the same
scene, view count and budget.

In [1]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd
from src import analysis as A

RESULTS = os.environ.get('RESULTS_CSV', '../results/results.csv')
FIG = '../report/figures'; os.makedirs(FIG, exist_ok=True)
df = A.load_results(RESULTS)
df.groupby('stage').size()

ModuleNotFoundError: No module named 'src'

## Reproduction check — vanilla 3DGS, Blender 8 views, DietNeRF split
Published (DNGaussian, CVPR'24, 3DGS row): **22.23 PSNR / 0.858 SSIM / 0.114 LPIPS**.

In [ ]:
rep = df[df.stage == 'stage0_repro']
display(rep.set_index('scene')[['psnr', 'ssim', 'lpips']].round(3))
print('mean', rep[['psnr', 'ssim', 'lpips']].mean().round(3).to_dict(),
      '| gap to 22.23 dB:', round(rep.psnr.mean() - 22.23, 2))

## Initialization vs. view count (no depth loss)

In [ ]:
s1 = df[df.stage == 'stage1']
display(A.summary(s1, ['init', 'n_views']))
sfm = s1[s1.init == 'sfm']
if len(sfm) and 'init_n_init_points' in sfm:
    print('SfM points triangulated per run:')
    display(sfm.pivot_table(index='scene', columns='n_views', values='init_n_init_points'))
A.save(A.plot_init_by_views(df), f'{FIG}/init_by_views')

## Appearance–geometry divergence
Every run against vanilla 3DGS, then neighbours along λ.

In [ ]:
print(df.effect.value_counts().to_string())
display(A.divergences(df).head(25))
display(A.axis_divergences(df, 'depth_lambda'))

## λ sweep — where depth flips from help to harm

In [ ]:
for n in sorted(df[df.stage == 'stage2_lambda'].n_views.unique()):
    A.save(A.plot_lambda_sweep(df, n_views=n), f'{FIG}/lambda_sweep_{n}v')
display(A.summary(df[df.stage.str.startswith('stage2')], ['init', 'n_views', 'depth_kind', 'depth_lambda']))

## Controlled degradation, and where the real model lands
x = the prior's unaligned AbsRel on the input views, so every corruption arm and Depth Anything V2 share one axis.

In [ ]:
for kind in ('ssi', 'absolute'):
    A.save(A.plot_degradation(df, kind=kind), f'{FIG}/degradation_{kind}')
real = df[df.depth_model != 'gt']
if len(real):
    cols = [c for c in ('prior_absrel', 'prior_aligned_absrel', 'depth_aligned_scale',
                        'depth_aligned_shift', 'psnr', 'geometry') if c in real]
    display(real.groupby(['depth_align', 'path', 'depth_kind'])[cols].mean().round(4))
else:
    print('no real-model runs yet (stage4_real)')

## The trade-off map

In [ ]:
A.save(A.plot_tradeoff_map(df), f'{FIG}/tradeoff_map')

## Convergence and cost
Iterations until the 4-view probe PSNR first reaches vanilla's final probe PSNR (NaN = never).

In [ ]:
df['iters_to_vanilla'] = A.iters_to_target(df)
cols = [c for c in ('iters_to_vanilla', 'n_gaussians', 'train_seconds') if c in df]
display(df.groupby(['stage', 'path'])[cols].mean().round(0))

## DTU realism check (SampleSet scans 1 and 6, official Chamfer in mm)

In [ ]:
dtu = df[df.dataset == 'dtu']
if len(dtu):
    display(A.summary(dtu, ['stage', 'init', 'depth_model', 'n_views', 'depth_lambda'],
                      metrics=('psnr', 'ssim', 'lpips', 'chamfer', 'chamfer_acc', 'chamfer_comp')))